In [1]:
import pandas as pd
import numpy as np
import random
import time

In [2]:
%matplotlib notebook

In [5]:
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import numpy as np
from IPython.display import HTML
import os

class Machine:
    def __init__(self, processing_times, setup_times):
        self.processing_times = processing_times
        self.setup_times = setup_times
        self.current_product = None
        self.finish_time = 0
        self.busy = False
        self.event_log = []

    def process(self, product, current_time):
        self.busy = True
        setup_time = 0
        if self.current_product is not None:
            setup_time = self.setup_times[self.current_product][product]
        start_time = current_time + setup_time
        self.finish_time = start_time + self.processing_times[product]
        self.event_log.append(('setup', current_time, start_time, setup_time, product, self.current_product))
        self.event_log.append((product, start_time, self.finish_time))
        self.current_product = product  # Update current_product after setup calculation
        return self.finish_time
    
    def update(self, current_time):
        if current_time >= self.finish_time:
            self.busy = False
            finished_product = self.current_product
            # Do not reset current_product here; let it change when a new product starts processing
            return finished_product
        return None

    def print_event_log(self):
        for event in self.event_log:
            if event[0] == 'setup':
                print(f"Setup for Product {event[4]} after {event[5]}: from {event[1]} to {event[2]} (Setup time: {event[3]} sec)")
            else:
                print(f"Product {event[0]}: Processed from {event[1]} to {event[2]}")


class ProductionLine:
    def __init__(self, machine1_times, machine1_setups, machine2_times, machine2_setups, arrival_rate1, arrival_rate2):
        self.machine1 = Machine(machine1_times, machine1_setups)
        self.machine2 = Machine(machine2_times, machine2_setups)
        self.arrival_rate1 = arrival_rate1  # Arrival rate for Product 0
        self.arrival_rate2 = arrival_rate2  # Arrival rate for Product 1
        self.next_arrival1 = 0  # Next arrival time for Product 0
        self.next_arrival2 = 0  # Next arrival time for Product 1
        self.queue = []
        self.finished_products = [0, 0]
        self.queue_log = []

    def add_item(self, product, current_time):
        self.queue.append((product, current_time))
        self.print_queue_status(current_time)  # Print queue status when an item is added

    def run_step(self, current_time):
        if self.queue and not self.machine1.busy:
            product, _ = self.queue.pop(0)
            self.machine1.process(product, current_time)
            self.print_queue_status(current_time)  # Optional: Print queue status after processing starts

        if not self.machine2.busy and self.machine1.busy and current_time >= self.machine1.finish_time:
            finished_product = self.machine1.update(current_time)
            if finished_product is not None:
                self.machine2.process(finished_product, current_time)

        finished_product = self.machine2.update(current_time)
        if finished_product is not None:
            self.finished_products[finished_product] += 1

        # Check for arrivals of Product 0
        if current_time >= self.next_arrival1:
            self.add_item(0, current_time)
            self.next_arrival1 += self.arrival_rate1

        # Check for arrivals of Product 1
        if current_time >= self.next_arrival2:
            self.add_item(1, current_time)
            self.next_arrival2 += self.arrival_rate2

    def print_queue_status(self, current_time):
        queue_status = f"Queue at Time {current_time}: ["
        queue_status += ", ".join(f"Product {prod} (Arrived at {time})" for prod, time in self.queue)
        queue_status += "]"
        print(queue_status)





# Create ProductionLine instance
production_line = ProductionLine({0: 15, 1: 10}, {None: {0: 5, 1: 3}, 0: {0: 0, 1: 10}, 1: {0: 5, 1: 0}},
                                 {0: 3, 1: 4}, {None: {0: 0, 1: 0}, 0: {0: 0, 1: 2}, 1: {0: 3, 1: 0}}, 10,8)
# Simulate the production line for 50 time units
for current_time in range(200):
    production_line.run_step(current_time)

# Print event logs of each machine
production_line.machine1.print_event_log()
production_line.machine2.print_event_log()


Queue at Time 0: [Product 0 (Arrived at 0)]
Queue at Time 0: [Product 0 (Arrived at 0), Product 1 (Arrived at 0)]
Queue at Time 1: [Product 1 (Arrived at 0)]
Queue at Time 8: [Product 1 (Arrived at 0), Product 1 (Arrived at 8)]
Queue at Time 10: [Product 1 (Arrived at 0), Product 1 (Arrived at 8), Product 0 (Arrived at 10)]
Queue at Time 16: [Product 1 (Arrived at 0), Product 1 (Arrived at 8), Product 0 (Arrived at 10), Product 1 (Arrived at 16)]
Queue at Time 17: [Product 1 (Arrived at 8), Product 0 (Arrived at 10), Product 1 (Arrived at 16)]
Queue at Time 20: [Product 1 (Arrived at 8), Product 0 (Arrived at 10), Product 1 (Arrived at 16), Product 0 (Arrived at 20)]
Queue at Time 24: [Product 1 (Arrived at 8), Product 0 (Arrived at 10), Product 1 (Arrived at 16), Product 0 (Arrived at 20), Product 1 (Arrived at 24)]
Queue at Time 30: [Product 1 (Arrived at 8), Product 0 (Arrived at 10), Product 1 (Arrived at 16), Product 0 (Arrived at 20), Product 1 (Arrived at 24), Product 0 (Arrived

In [ ]:
#Next steps
# 1) Introduce a demand forecast into the state space and reward it for meeting future excess demand
# 2) See if it can switch demand patterns quickly 
# 3) Q-learning implementaion to achive the same result
# 4) Write a LP program of this